# Notebook 3: Model Implementation
**DS340W Capstone — Group 60**

Builds and trains the Informer + ST-GCN hybrid model with STL decomposition enhancement.

In [ ]:
# Cell 1: Setup
from google.colab import drive
drive.mount('/content/drive')
!pip install torch torchvision -q
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset
import warnings; warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
base_path = '/content/drive/MyDrive/capstone_project'

In [ ]:
# Cell 2: Load data
train_df = pd.read_csv(f'{base_path}/processed_data/train_data.csv')
test_df = pd.read_csv(f'{base_path}/processed_data/test_data.csv')
adj_matrix = np.load(f'{base_path}/processed_data/adjacency_matrix.npy')
precinct_ids = np.load(f'{base_path}/processed_data/precinct_ids.npy')
train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

N_PRECINCTS = len(precinct_ids)
CRIME_TYPES = ['assault', 'criminal_damage', 'robbery', 'theft']
FEATURE_COLS = ['weekend', 'holiday', 'temperature', 'humidity', 'wind_speed', 'precipitation', 'discomfort_index', 'apparent_temp']
train_df = train_df.sort_values(['date', 'precinct']).reset_index(drop=True)
test_df = test_df.sort_values(['date', 'precinct']).reset_index(drop=True)
train_dates = sorted(train_df['date'].unique())
test_dates = sorted(test_df['date'].unique())
print(f"Precincts: {N_PRECINCTS}, Train days: {len(train_dates)}, Test days: {len(test_dates)}")

In [ ]:
# Cell 3: Build tensors
def df_to_tensor(df, dates, precinct_ids, feature_cols, crime_types):
    n_days, n_precincts = len(dates), len(precinct_ids)
    crime_array = np.zeros((n_days, n_precincts, len(crime_types)))
    feat_array = np.zeros((n_days, n_precincts, len(feature_cols)))
    df_indexed = df.set_index(['date', 'precinct'])
    for d_idx, date in enumerate(dates):
        if (d_idx+1) % 200 == 0: print(f"  Day {d_idx+1}/{n_days}...")
        for p_idx, pct in enumerate(precinct_ids):
            try:
                row = df_indexed.loc[(date, pct)]
                for c_idx, crime in enumerate(crime_types): crime_array[d_idx, p_idx, c_idx] = row[crime]
                for f_idx, feat in enumerate(feature_cols):
                    val = row[feat]; feat_array[d_idx, p_idx, f_idx] = val if not pd.isna(val) else 0
            except KeyError: pass
    return crime_array, feat_array

print("Building training tensors...")
train_crimes, train_feats = df_to_tensor(train_df, train_dates, precinct_ids, FEATURE_COLS, CRIME_TYPES)
print("Building test tensors...")
test_crimes, test_feats = df_to_tensor(test_df, test_dates, precinct_ids, FEATURE_COLS, CRIME_TYPES)

scaler = MinMaxScaler()
train_feats = scaler.fit_transform(train_feats.reshape(-1, train_feats.shape[-1])).reshape(train_feats.shape)
test_feats = scaler.transform(test_feats.reshape(-1, test_feats.shape[-1])).reshape(test_feats.shape)
print(f"Train: {train_crimes.shape}, Test: {test_crimes.shape}")

In [ ]:
# Cell 4: Create sequences
LOOKBACK = 7
def create_sequences(crimes, features, lookback):
    X_crime, X_feat, Y = [], [], []
    for i in range(lookback, crimes.shape[0]):
        X_crime.append(crimes[i-lookback:i]); X_feat.append(features[i-lookback:i]); Y.append(crimes[i])
    return np.array(X_crime), np.array(X_feat), np.array(Y)

X_crime_train, X_feat_train, Y_train = create_sequences(train_crimes, train_feats, LOOKBACK)
X_crime_tensor = torch.FloatTensor(X_crime_train).to(device)
X_feat_tensor = torch.FloatTensor(X_feat_train).to(device)
Y_tensor = torch.FloatTensor(Y_train).to(device)
print(f"Sequences: {X_crime_train.shape[0]} samples")

In [ ]:
# Cell 5: GCN Layer
class GCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
    def forward(self, X, A_hat):
        return F.relu(self.linear(torch.matmul(A_hat, X)))

A = torch.FloatTensor(adj_matrix).to(device)
D_inv_sqrt = torch.diag(1.0 / torch.sqrt(torch.sum(A, dim=1)))
D_inv_sqrt[D_inv_sqrt == float('inf')] = 0
A_hat = torch.matmul(torch.matmul(D_inv_sqrt, A), D_inv_sqrt)
print(f"A_hat: {A_hat.shape}")

In [ ]:
# Cell 6: ST-GCN Module
class STGCNBlock(nn.Module):
    def __init__(self, in_ch, hid_ch, out_ch):
        super().__init__()
        self.gcn1 = GCNLayer(in_ch, hid_ch); self.gcn2 = GCNLayer(hid_ch, out_ch)
        self.residual = nn.Linear(in_ch, out_ch) if in_ch != out_ch else nn.Identity()
    def forward(self, X, A_hat):
        return F.relu(self.gcn2(self.gcn1(X, A_hat), A_hat) + self.residual(X))

class STGCN(nn.Module):
    def __init__(self, in_ch, hid=64, out=128):
        super().__init__()
        self.prox = STGCNBlock(in_ch, hid, out); self.period = STGCNBlock(in_ch, hid, out)
        self.trend = STGCNBlock(in_ch, hid, out); self.fusion = nn.Linear(out*3, out)
    def forward(self, X_p, X_pe, X_t, A_hat):
        return F.relu(self.fusion(torch.cat([self.prox(X_p, A_hat), self.period(X_pe, A_hat), self.trend(X_t, A_hat)], dim=-1)))
print("ST-GCN defined")

In [ ]:
# Cell 7: Informer Module
class InformerEncoder(nn.Module):
    def __init__(self, input_dim, d_model=64, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        self.positional_encoding = nn.Parameter(torch.randn(1, 500, d_model) * 0.01)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.distill = nn.Sequential(nn.Conv1d(d_model, d_model, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool1d(kernel_size=2, stride=1, padding=0))
        self.output_dim = d_model
    def forward(self, X):
        h = self.input_projection(X) + self.positional_encoding[:, :X.shape[1], :]
        h = self.transformer(h)
        return self.distill(h.permute(0,2,1))[:, :, -1]
print("Informer defined")

In [ ]:
# Cell 8: Full Hybrid Model
class CrimePredictionModel(nn.Module):
    def __init__(self, n_precincts, n_crime=4, n_feat=8, lookback=7, hid=64, stgcn_out=128, inf_d=64):
        super().__init__()
        self.n_precincts, self.lookback = n_precincts, lookback
        self.stgcn = STGCN(n_crime+n_feat, hid, stgcn_out)
        self.informer = InformerEncoder(n_crime+n_feat, d_model=inf_d)
        self.fc1 = nn.Linear(stgcn_out+inf_d, hid); self.fc2 = nn.Linear(hid, n_crime)
        self.dropout = nn.Dropout(0.1)
    def forward(self, X_crime, X_feat, A_hat):
        B = X_crime.shape[0]
        X = torch.cat([X_crime, X_feat], dim=-1)
        stg = self.stgcn(X[:,-1,:,:], X[:,-3,:,:], X[:,0,:,:], A_hat)
        X_inf = X.permute(0,2,1,3).reshape(B*self.n_precincts, self.lookback, -1)
        inf = self.informer(X_inf).reshape(B, self.n_precincts, -1)
        return self.fc2(self.dropout(F.relu(self.fc1(torch.cat([stg, inf], dim=-1)))))

model = CrimePredictionModel(N_PRECINCTS, n_feat=len(FEATURE_COLS), lookback=LOOKBACK).to(device)
print(f"Model: {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
# Cell 9: Train
dataset = TensorDataset(X_crime_tensor, X_feat_tensor, Y_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
optimizer = torch.optim.NAdam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
criterion = nn.MSELoss()

print("Training original model...")
best_loss, patience_counter, train_losses = float('inf'), 0, []
for epoch in range(100):
    model.train(); epoch_loss, n = 0, 0
    for bc, bf, by in dataloader:
        optimizer.zero_grad(); loss = criterion(model(bc, bf, A_hat), by)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        epoch_loss += loss.item(); n += 1
    scheduler.step(); avg = epoch_loss/n; train_losses.append(avg)
    if avg < best_loss: best_loss = avg; patience_counter = 0; torch.save(model.state_dict(), f'{base_path}/processed_data/best_model.pth')
    else: patience_counter += 1
    if (epoch+1) % 10 == 0: print(f"  Epoch {epoch+1} | Loss: {avg:.4f} | Best: {best_loss:.4f} | Patience: {patience_counter}/20")
    if patience_counter >= 20: print(f"Early stopping at epoch {epoch+1}"); break
print(f"Training complete! Best loss: {best_loss:.4f}")

plt.figure(figsize=(10,4)); plt.plot(train_losses); plt.title('Training Loss'); plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.savefig(f'{base_path}/outputs/training_loss.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Cell 10: Predict (original model)
model.load_state_dict(torch.load(f'{base_path}/processed_data/best_model.pth')); model.eval()
combined_c = np.concatenate([train_crimes[-LOOKBACK:], test_crimes])
combined_f = np.concatenate([train_feats[-LOOKBACK:], test_feats])
test_predictions = []
with torch.no_grad():
    for i in range(LOOKBACK, len(combined_c)):
        pred = model(torch.FloatTensor(combined_c[i-LOOKBACK:i]).unsqueeze(0).to(device),
                     torch.FloatTensor(combined_f[i-LOOKBACK:i]).unsqueeze(0).to(device), A_hat)
        test_predictions.append(pred.cpu().numpy()[0])
test_predictions = np.maximum(np.array(test_predictions), 0)
test_actual = test_crimes

for c_idx, crime in enumerate(CRIME_TYPES):
    mae = np.mean(np.abs(test_actual[:,:,c_idx] - test_predictions[:,:,c_idx]))
    print(f"  {crime:20s} MAE: {mae:.2f}")

np.save(f'{base_path}/processed_data/test_predictions.npy', test_predictions)
np.save(f'{base_path}/processed_data/test_actual.npy', test_actual)
print("Original predictions saved!")

## Novel Enhancement: STL Decomposition
Inspired by Butt et al. (START), we apply STL decomposition to remove weekly seasonality before model input.

In [ ]:
# Cell 11: STL Decomposition
from statsmodels.tsa.seasonal import STL

print("Applying STL decomposition...")
train_crimes_stl = np.zeros_like(train_crimes)
for p_idx in range(len(precinct_ids)):
    for c_idx in range(4):
        series = train_crimes[:, p_idx, c_idx]
        try:
            result = STL(series, period=7, robust=True).fit()
            train_crimes_stl[:, p_idx, c_idx] = result.trend + result.resid
        except:
            train_crimes_stl[:, p_idx, c_idx] = series
    if (p_idx+1) % 10 == 0: print(f"  {p_idx+1}/{len(precinct_ids)} precincts")

test_crimes_stl = np.zeros_like(test_crimes)
for p_idx in range(len(precinct_ids)):
    for c_idx in range(4):
        combined = np.concatenate([train_crimes[-60:, p_idx, c_idx], test_crimes[:, p_idx, c_idx]])
        try:
            result = STL(combined, period=7, robust=True).fit()
            test_crimes_stl[:, p_idx, c_idx] = (result.trend + result.resid)[-7:]
        except:
            test_crimes_stl[:, p_idx, c_idx] = test_crimes[:, p_idx, c_idx]
print("STL decomposition complete!")

In [ ]:
# Cell 12: Train STL model
X_cs, X_fs, Y_s = create_sequences(train_crimes_stl, train_feats, LOOKBACK)
model_stl = CrimePredictionModel(N_PRECINCTS, n_feat=len(FEATURE_COLS), lookback=LOOKBACK).to(device)
ds = TensorDataset(torch.FloatTensor(X_cs).to(device), torch.FloatTensor(X_fs).to(device), torch.FloatTensor(Y_s).to(device))
dl = DataLoader(ds, batch_size=32, shuffle=True)
opt = torch.optim.NAdam(model_stl.parameters(), lr=0.001)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)

print("Training STL model...")
best, pat = float('inf'), 0
for epoch in range(100):
    model_stl.train(); el, n = 0, 0
    for bc, bf, by in dl:
        opt.zero_grad(); loss = criterion(model_stl(bc, bf, A_hat), by)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model_stl.parameters(), 1.0); opt.step()
        el += loss.item(); n += 1
    sch.step(); avg = el/n
    if avg < best: best = avg; pat = 0; torch.save(model_stl.state_dict(), f'{base_path}/processed_data/best_model_stl.pth')
    else: pat += 1
    if (epoch+1) % 10 == 0: print(f"  Epoch {epoch+1} | Loss: {avg:.4f} | Best: {best:.4f}")
    if pat >= 20: print(f"Early stopping at epoch {epoch+1}"); break
print(f"STL model complete! Best loss: {best:.4f}")

In [ ]:
# Cell 13: Compare original vs STL
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model_stl.load_state_dict(torch.load(f'{base_path}/processed_data/best_model_stl.pth')); model_stl.eval()
combined_cs = np.concatenate([train_crimes_stl[-LOOKBACK:], test_crimes_stl])
combined_fs = np.concatenate([train_feats[-LOOKBACK:], test_feats])
preds_stl = []
with torch.no_grad():
    for i in range(LOOKBACK, len(combined_cs)):
        pred = model_stl(torch.FloatTensor(combined_cs[i-LOOKBACK:i]).unsqueeze(0).to(device),
                         torch.FloatTensor(combined_fs[i-LOOKBACK:i]).unsqueeze(0).to(device), A_hat)
        preds_stl.append(pred.cpu().numpy()[0])
preds_stl = np.maximum(np.array(preds_stl), 0)

print("=" * 75)
print("COMPARISON: Original vs STL Enhanced")
print("=" * 75)
for c_idx, crime in enumerate(CRIME_TYPES):
    y = test_actual[:,:,c_idx].flatten()
    mae_o = mean_absolute_error(y, test_predictions[:,:,c_idx].flatten())
    mae_s = mean_absolute_error(y, preds_stl[:,:,c_idx].flatten())
    change = ((mae_s - mae_o) / mae_o) * 100
    print(f"  {crime:20s} Original: {mae_o:.2f}  STL: {mae_s:.2f}  Change: {change:+.1f}%")

np.save(f'{base_path}/processed_data/test_predictions_stl.npy', preds_stl)
print("\nSTL predictions saved!")